In [1]:
import os
print(os.getcwd())

c:\Users\gonca\OneDrive\Documentos\IST\MECD\VI\renewable-energy-visualization\data-processing\src


In [2]:
import pandas as pd

df_energy = pd.read_csv("./release_generation_yearly_lower.csv")
df_prices = pd.read_csv("./eurostat_prices.csv")

In [3]:
print(df_energy.columns)

Index(['Area', 'ISO 3 code', 'Year', 'Area type', 'Electricity source',
       'Is aggregated source', 'Generation (TWh)',
       'Generation YoY change (TWh)', 'Generation YoY change (%)',
       'Share of generation (%)', 'Share of generation YoY change (% points)',
       'Emissions (MtCO2e)', 'Emissions YoY change (MtCO2e)',
       'Emissions YoY change (%)', 'Share of emissions (%)',
       'Emissions intensity (gCO2e/kWh)', 'Continent', 'Ember region',
       'EU member', 'OECD member', 'G20 member', 'G7 member', 'ASEAN member'],
      dtype='str')


In [4]:
keep_cols = [
    "Area", "ISO 3 code", "Year", "Electricity source", "Is aggregated source",
    "Generation (TWh)", "Share of generation (%)",'Emissions (MtCO2e)','Share of emissions (%)','Emissions intensity (gCO2e/kWh)'
]

df_energy = df_energy[keep_cols].rename(columns={
    "Area": "country", "ISO 3 code": "iso3", "Year": "year",
    "Electricity source": "source", "Is aggregated source": "is_aggregated",
    "Generation (TWh)": "generation_twh", "Share of generation (%)": "share_pct",
})

In [5]:
keep_cols2 = [
    "Geopolitical entity (reporting)", "TIME_PERIOD","OBS_VALUE"
]

df_prices = df_prices[keep_cols2].rename(columns={
    "Geopolitical entity (reporting)": "country", "TIME_PERIOD": "time", "OBS_VALUE": "price_eur_mwh"
})

print(df_prices.columns)

Index(['country', 'time', 'price_eur_mwh'], dtype='str')


In [30]:
df_prices[['year', 'semester']] = df_prices['time'].str.split('-', expand=True)
df_prices['year'] = df_prices['year'].astype(int)
df_prices['semester'] = df_prices['semester'].str.replace('S', '').astype(int)
df_prices = df_prices.drop(columns=['time'])

df_prices = df_prices[(df_prices["year"] >= 2016) & (df_prices["year"] <= 2025)].copy()

df_prices = df_prices.sort_values(["country", "year", "semester"]).reset_index(drop=True)

In [6]:
df_prices['country'] = df_prices['country'].replace('Kosovo*', 'Kosovo')
df_energy['country'] = df_energy['country'].replace('Bosnia Herzegovina', 'Bosnia and Herzegovina')

In [32]:
df_energy['country'].unique()

<StringArray>
[               'Austria',                'Belgium', 'Bosnia and Herzegovina',
               'Bulgaria',                'Croatia',                 'Cyprus',
                'Czechia',                'Denmark',                     'EU',
                'Estonia',                'Finland',                 'France',
                'Germany',                 'Greece',                'Hungary',
                'Ireland',                  'Italy',                 'Kosovo',
                 'Latvia',              'Lithuania',             'Luxembourg',
                  'Malta',             'Montenegro',            'Netherlands',
        'North Macedonia',                 'Norway',                 'Poland',
               'Portugal',                'Romania',                 'Serbia',
               'Slovakia',               'Slovenia',                  'Spain',
                 'Sweden',            'Switzerland',                'Türkiye',
                'Ukraine',         'Un

In [33]:
df_prices['country'].unique()

<StringArray>
[                                                                                                                 'Albania',
                                                                                                                  'Austria',
                                                                                                                  'Belgium',
                                                                                                   'Bosnia and Herzegovina',
                                                                                                                 'Bulgaria',
                                                                                                                  'Croatia',
                                                                                                                   'Cyprus',
                                                                                                               

In [34]:
energy_countries = set(df_energy['country'].unique())
price_countries = set(df_prices['country'].unique())

print("Energy only:", sorted(energy_countries - price_countries))
print("Prices only:", sorted(price_countries - energy_countries))

Energy only: ['EU', 'Switzerland']
Prices only: ['Albania', 'Euro area (EA11-1999, EA12-2001, EA13-2007, EA15-2008, EA16-2009, EA17-2011, EA18-2014, EA19-2015, EA20-2023, EA21-2026)', 'European Union - 27 countries (from 2020)', 'Georgia', 'Iceland', 'Liechtenstein', 'Moldova']


In [35]:
df_prices = df_prices[df_prices['country'].isin(energy_countries)]

In [36]:
df_prices.to_csv("../export/prices.csv", index=False)

In [37]:
df_energy['source'].unique()

<StringArray>
[                                'Solar',
                          'Onshore wind',
                                 'Hydro',
                             'Bioenergy',
                      'Other renewables',
                               'Nuclear',
                               'Lignite',
                                   'Gas',
                          'Other fossil',
                           'Net imports',
                                  'Wind',
                                  'Coal',
                        'Wind and solar',
 'Hydro, bioenergy and other renewables',
                            'Renewables',
                                 'Clean',
                                'Fossil',
                      'Total generation',
                                'Demand',
                               'Thermal',
                             'Hard coal',
                         'Offshore wind']
Length: 22, dtype: str

In [7]:
keep_sources = [
    "Total generation", "Net imports", "Demand",
    "Clean", "Fossil", "Renewables", "Nuclear",
    "Wind", "Solar", "Hydro", "Bioenergy", "Other renewables",
    "Coal", "Gas", "Other fossil",
]

df_energy = df_energy[df_energy["source"].isin(keep_sources)].copy()

In [39]:
df_energy = df_energy.sort_values(["country", "year", "source"]).reset_index(drop=True)
df_energy.to_csv("../export/generation_by_source.csv", index=False)

In [40]:
first_year_per_country = df_energy.groupby("country")["year"].min().sort_values(ascending=False)

print(first_year_per_country)
print()
print("Highest first year (i.e. shortest history):", first_year_per_country.index[0], "-", first_year_per_country.iloc[0])

country
Montenegro                2005
Bosnia and Herzegovina    2000
Kosovo                    2000
Switzerland               2000
EU                        2000
Austria                   1990
Cyprus                    1990
Croatia                   1990
Bulgaria                  1990
Belgium                   1990
Finland                   1990
France                    1990
Denmark                   1990
Czechia                   1990
Latvia                    1990
Hungary                   1990
Ireland                   1990
Italy                     1990
Luxembourg                1990
Lithuania                 1990
Greece                    1990
Estonia                   1990
Serbia                    1990
Malta                     1990
North Macedonia           1990
Netherlands               1990
Poland                    1990
Portugal                  1990
Romania                   1990
Norway                    1990
Sweden                    1990
Slovakia                  1990


In [41]:
last_year_per_country = df_energy.groupby("country")["year"].max().sort_values(ascending=True)

print(last_year_per_country)
print()
print("Lowest last year (first to end):", last_year_per_country.index[0], "-", last_year_per_country.iloc[0])

country
Ukraine                   2022
Austria                   2025
Bosnia and Herzegovina    2025
Belgium                   2025
Croatia                   2025
Cyprus                    2025
Czechia                   2025
Bulgaria                  2025
EU                        2025
Estonia                   2025
Finland                   2025
France                    2025
Germany                   2025
Greece                    2025
Hungary                   2025
Denmark                   2025
Italy                     2025
Kosovo                    2025
Latvia                    2025
Lithuania                 2025
Luxembourg                2025
Malta                     2025
Montenegro                2025
Netherlands               2025
North Macedonia           2025
Norway                    2025
Poland                    2025
Portugal                  2025
Romania                   2025
Serbia                    2025
Slovakia                  2025
Ireland                   2025


In [42]:
print(df_prices.columns)

Index(['country', 'price_eur_mwh', 'year', 'semester'], dtype='str')


In [43]:
print(df_prices["country"].unique())

<StringArray>
[               'Austria',                'Belgium', 'Bosnia and Herzegovina',
               'Bulgaria',                'Croatia',                 'Cyprus',
                'Czechia',                'Denmark',                'Estonia',
                'Finland',                 'France',                'Germany',
                 'Greece',                'Hungary',                'Ireland',
                  'Italy',                 'Kosovo',                 'Latvia',
              'Lithuania',             'Luxembourg',                  'Malta',
             'Montenegro',            'Netherlands',        'North Macedonia',
                 'Norway',                 'Poland',               'Portugal',
                'Romania',                 'Serbia',               'Slovakia',
               'Slovenia',                  'Spain',                 'Sweden',
                'Türkiye',                'Ukraine',         'United Kingdom']
Length: 36, dtype: str


In [44]:
first_year_per_country = df_prices.groupby("country")["year"].min().sort_values(ascending=False)

print(first_year_per_country)
print()
print("Highest first year (i.e. shortest history):", first_year_per_country.index[0], "-", first_year_per_country.iloc[0])

country
Austria                   2016
Belgium                   2016
Bosnia and Herzegovina    2016
Bulgaria                  2016
Croatia                   2016
Cyprus                    2016
Czechia                   2016
Denmark                   2016
Estonia                   2016
Finland                   2016
France                    2016
Germany                   2016
Greece                    2016
Hungary                   2016
Ireland                   2016
Italy                     2016
Kosovo                    2016
Latvia                    2016
Lithuania                 2016
Luxembourg                2016
Malta                     2016
Montenegro                2016
Netherlands               2016
North Macedonia           2016
Norway                    2016
Poland                    2016
Portugal                  2016
Romania                   2016
Serbia                    2016
Slovakia                  2016
Slovenia                  2016
Spain                     2016


In [45]:
last_year_per_country = df_prices.groupby("country")["year"].max().sort_values(ascending=True)

print(last_year_per_country)
print()
print("Lowest last year (first to end):", last_year_per_country.index[0], "-", last_year_per_country.iloc[0])

country
United Kingdom            2020
Ukraine                   2021
Austria                   2025
Belgium                   2025
Croatia                   2025
Cyprus                    2025
Bosnia and Herzegovina    2025
Bulgaria                  2025
Estonia                   2025
Finland                   2025
France                    2025
Germany                   2025
Greece                    2025
Hungary                   2025
Czechia                   2025
Denmark                   2025
Kosovo                    2025
Latvia                    2025
Lithuania                 2025
Luxembourg                2025
Malta                     2025
Montenegro                2025
Netherlands               2025
North Macedonia           2025
Norway                    2025
Poland                    2025
Portugal                  2025
Romania                   2025
Serbia                    2025
Slovakia                  2025
Ireland                   2025
Italy                     2025


In [46]:
df_energy['country'].unique()

<StringArray>
[               'Austria',                'Belgium', 'Bosnia and Herzegovina',
               'Bulgaria',                'Croatia',                 'Cyprus',
                'Czechia',                'Denmark',                     'EU',
                'Estonia',                'Finland',                 'France',
                'Germany',                 'Greece',                'Hungary',
                'Ireland',                  'Italy',                 'Kosovo',
                 'Latvia',              'Lithuania',             'Luxembourg',
                  'Malta',             'Montenegro',            'Netherlands',
        'North Macedonia',                 'Norway',                 'Poland',
               'Portugal',                'Romania',                 'Serbia',
               'Slovakia',               'Slovenia',                  'Spain',
                 'Sweden',            'Switzerland',                'Türkiye',
                'Ukraine',         'Un

In [9]:
df_energy = df_energy[(df_energy['year'] >= 2005) & (df_energy['year'] <= 2025)].reset_index(drop=True)

numeric_cols = [
    'generation_twh', 'share_pct', 'emissions_mtco2e',
    'share_emissions_pct', 'emissions_intensity_gco2e_kwh'
]

eu_accession_year = {
    'Belgium': 1958, 'France': 1958, 'Germany': 1958, 'Italy': 1958,
    'Luxembourg': 1958, 'Netherlands': 1958,
    'Denmark': 1973, 'Ireland': 1973,
    'Greece': 1981,
    'Portugal': 1986, 'Spain': 1986,
    'Austria': 1995, 'Finland': 1995, 'Sweden': 1995,
    'Cyprus': 2004, 'Czechia': 2004, 'Estonia': 2004, 'Hungary': 2004,
    'Latvia': 2004, 'Lithuania': 2004, 'Malta': 2004, 'Poland': 2004,
    'Slovakia': 2004, 'Slovenia': 2004,
    'Bulgaria': 2007, 'Romania': 2007,
    'Croatia': 2013,
}
uk_membership = (1973, 2020)  # inclusive start, exclusive end

def is_eu_member(country, year):
    if country == 'United Kingdom':
        return uk_membership[0] <= year < uk_membership[1]
    return eu_accession_year.get(country, 9999) <= year

all_countries = [c for c in df_energy['country'].unique() if c not in ['EU', 'EU2', 'Non-EU']]

def aggregate_region(df, region_name, membership_fn):
    base = df[df['country'].isin(all_countries)].copy()
    is_member = base.apply(lambda r: membership_fn(r['country'], r['year']), axis=1)
    subset = base[is_member]

    grouped = subset.groupby(['year', 'source'], as_index=False).agg(
        generation_twh=('generation_twh', 'sum'),
        emissions_mtco2e=('emissions_mtco2e', 'sum')
    )

    totals = grouped[grouped['source'] == 'Total generation'][
        ['year', 'generation_twh', 'emissions_mtco2e']
    ].rename(columns={'generation_twh': 'total_gen', 'emissions_mtco2e': 'total_emissions'})

    grouped = grouped.merge(totals, on='year', how='left')
    grouped['share_pct'] = grouped['generation_twh'] / grouped['total_gen'] * 100
    grouped['share_emissions_pct'] = grouped['emissions_mtco2e'] / grouped['total_emissions'] * 100
    grouped['emissions_intensity_gco2e_kwh'] = grouped['emissions_mtco2e'] / grouped['generation_twh'] * 1000

    grouped['country'] = region_name
    grouped['iso3'] = None
    grouped['is_aggregated'] = True
    grouped = grouped.drop(columns=['total_gen', 'total_emissions'])
    return grouped[df.columns]

eu2 = aggregate_region(df_energy, 'EU2', is_eu_member)
non_eu = aggregate_region(df_energy, 'Non-EU', lambda c, y: not is_eu_member(c, y))

df_energy = df_energy[~df_energy['country'].isin(['EU2', 'Non-EU'])]
df_energy = pd.concat([df_energy, eu2, non_eu], ignore_index=True)

comparison = (
    df_energy[df_energy['country'].isin(['EU', 'EU2'])]
    .pivot_table(index=['year', 'source'], columns='country', values=numeric_cols)
)
for col in numeric_cols:
    sub = comparison[col].copy()
    sub['abs_diff'] = (sub['EU'] - sub['EU2']).abs()
    sub['pct_diff'] = (sub['abs_diff'] / sub['EU'].abs()) * 100
    print(f"\n--- {col} ---")
    print(sub.sort_values('pct_diff', ascending=False).head(5))



--- generation_twh ---
country             EU    EU2  abs_diff     pct_diff
year source                                         
2006 Net imports  0.46  14.99     14.53  3158.695652
2016 Net imports  0.65  18.40     17.75  2730.769231
2013 Net imports -1.81  12.62     14.43   797.237569
2019 Net imports  2.93  24.10     21.17   722.525597
2014 Net imports -5.03  15.49     20.52   407.952286

--- share_pct ---
country              EU       EU2  abs_diff     pct_diff
year source                                             
2006 Net imports  0.016  0.467172  0.451172  2819.823478
2016 Net imports  0.022  0.569557  0.547557  2488.896224
2013 Net imports -0.063  0.389269  0.452269   717.888075
2019 Net imports  0.102  0.752727  0.650727   637.968104
2014 Net imports -0.178  0.489881  0.667881   375.214254

--- emissions_mtco2e ---
country           EU      EU2  abs_diff   pct_diff
year source                                       
2016 Gas     210.498  279.134    68.636  32.606486
2010 Gas

In [ ]:
df_energy = df_energy[(df_energy['year'] >= 2005) & (df_energy['year'] <= 2025)].reset_index(drop=True)


numeric_cols = [
    'generation_twh', 'share_pct', 'Emissions (MtCO2e)',
    'Share of emissions (%)', 'Emissions intensity (gCO2e/kWh)'
]

eu_accession_year = {
    'Belgium': 1958, 'France': 1958, 'Germany': 1958, 'Italy': 1958,
    'Luxembourg': 1958, 'Netherlands': 1958,
    'Denmark': 1973, 'Ireland': 1973,
    'Greece': 1981,
    'Portugal': 1986, 'Spain': 1986,
    'Austria': 1995, 'Finland': 1995, 'Sweden': 1995,
    'Cyprus': 2004, 'Czechia': 2004, 'Estonia': 2004, 'Hungary': 2004,
    'Latvia': 2004, 'Lithuania': 2004, 'Malta': 2004, 'Poland': 2004,
    'Slovakia': 2004, 'Slovenia': 2004,
    'Bulgaria': 2007, 'Romania': 2007,
    'Croatia': 2013,
}
uk_membership = (1973, 2020)  # inclusive start, exclusive end

def is_eu_member(country, year):
    if country == 'United Kingdom':
        return uk_membership[0] <= year < uk_membership[1]
    return eu_accession_year.get(country, 9999) <= year

all_countries = [c for c in df_energy['country'].unique() if c not in ['EU', 'EU2', 'Non-EU']]

def aggregate_region(df, region_name, membership_fn):
    base = df[df['country'].isin(all_countries)].copy()
    is_member = base.apply(lambda r: membership_fn(r['country'], r['year']), axis=1)
    subset = base[is_member]

    grouped = subset.groupby(['year', 'source'], as_index=False).agg(
        generation_twh=('generation_twh', 'sum'),
        emissions_mtco2e=('Emissions (MtCO2e)', 'sum')
    )

    totals = grouped[grouped['source'] == 'Total generation'][
        ['year', 'generation_twh', 'emissions_mtco2e']
    ].rename(columns={'generation_twh': 'total_gen', 'emissions_mtco2e': 'total_emissions'})

    grouped = grouped.merge(totals, on='year', how='left')
    grouped['share_pct'] = grouped['generation_twh'] / grouped['total_gen'] * 100
    grouped['Share of emissions (%)'] = grouped['emissions_mtco2e'] / grouped['total_emissions'] * 100
    grouped['Emissions intensity (gCO2e/kWh)'] = grouped['emissions_mtco2e'] / grouped['generation_twh'] * 1000

    grouped['country'] = region_name
    grouped['iso3'] = None
    grouped['is_aggregated'] = True
    grouped = grouped.rename(columns={'emissions_mtco2e': 'Emissions (MtCO2e)'})
    grouped = grouped.drop(columns=['total_gen', 'total_emissions'])
    return grouped[df.columns]

eu2 = aggregate_region(df_energy, 'EU2', is_eu_member)
non_eu = aggregate_region(df_energy, 'Non-EU', lambda c, y: not is_eu_member(c, y))

df_energy = df_energy[~df_energy['country'].isin(['EU2', 'Non-EU'])]
df_energy = pd.concat([df_energy, eu2, non_eu], ignore_index=True)

comparison = (
    df_energy[df_energy['country'].isin(['EU', 'EU2'])]
    .pivot_table(index=['year', 'source'], columns='country', values=numeric_cols)
)
for col in numeric_cols:
    sub = comparison[col].copy()
    sub['abs_diff'] = (sub['EU'] - sub['EU2']).abs()
    sub['pct_diff'] = (sub['abs_diff'] / sub['EU'].abs()) * 100
    print(f"\n--- {col} ---")
    print(sub.sort_values('pct_diff', ascending=False).head(5))


--- generation_twh ---
country             EU    EU2  abs_diff     pct_diff
year source                                         
2006 Net imports  0.46  14.99     14.53  3158.695652
2016 Net imports  0.65  18.40     17.75  2730.769231
2013 Net imports -1.81  12.62     14.43   797.237569
2019 Net imports  2.93  24.10     21.17   722.525597
2014 Net imports -5.03  15.49     20.52   407.952286

--- share_pct ---
country              EU       EU2  abs_diff     pct_diff
year source                                             
2006 Net imports  0.016  0.467172  0.451172  2819.823478
2016 Net imports  0.022  0.569557  0.547557  2488.896224
2013 Net imports -0.063  0.389269  0.452269   717.888075
2019 Net imports  0.102  0.752727  0.650727   637.968104
2014 Net imports -0.178  0.489881  0.667881   375.214254

--- Emissions (MtCO2e) ---
country           EU      EU2  abs_diff   pct_diff
year source                                       
2016 Gas     210.498  279.134    68.636  32.606486
2010 G

In [42]:
import pandas as pd
df_energy = pd.read_csv("./release_generation_yearly_lower.csv")

keep_cols_energy = [
    "Area", "ISO 3 code", "Year", 
    "Electricity source", "Is aggregated source","Generation (TWh)", "Share of generation (%)",
    'Emissions (MtCO2e)','Share of emissions (%)','Emissions intensity (gCO2e/kWh)'
]
df_energy = df_energy[keep_cols_energy].rename(columns={
    "Area": "country", "ISO 3 code": "iso3", "Year": "year",
    "Electricity source": "source", "Is aggregated source": "is_aggregated",
    "Generation (TWh)": "generation_twh", "Share of generation (%)": "share_pct",
    "Emissions (MtCO2e)": "emissions_mtco2e",
    "Share of emissions (%)": "share_emissions_pct",
    "Emissions intensity (gCO2e/kWh)": "emissions_intensity_gco2e_kwh",
})

keep_sources = [
    "Total generation", "Net imports", "Demand",
    "Clean", "Fossil", "Renewables", "Nuclear",
    "Wind", "Solar", "Hydro", "Bioenergy", "Other renewables",
    "Coal", "Gas", "Other fossil",
]

df_energy['country'] = df_energy['country'].replace('Bosnia Herzegovina', 'Bosnia and Herzegovina')

df_energy = df_energy[df_energy["source"].isin(keep_sources)].copy()

df_energy = df_energy[(df_energy['year'] >= 2005) & (df_energy['year'] <= 2025)].reset_index(drop=True)

numeric_cols = [
    'generation_twh', 'share_pct', 'emissions_mtco2e',
    'share_emissions_pct', 'emissions_intensity_gco2e_kwh'
]

eu_accession_year = {
    'Belgium': 1958, 'France': 1958, 'Germany': 1958, 'Italy': 1958,
    'Luxembourg': 1958, 'Netherlands': 1958,
    'Denmark': 1973, 'Ireland': 1973,
    'Greece': 1981,
    'Portugal': 1986, 'Spain': 1986,
    'Austria': 1995, 'Finland': 1995, 'Sweden': 1995,
    'Cyprus': 2004, 'Czechia': 2004, 'Estonia': 2004, 'Hungary': 2004,
    'Latvia': 2004, 'Lithuania': 2004, 'Malta': 2004, 'Poland': 2004,
    'Slovakia': 2004, 'Slovenia': 2004,
    'Bulgaria': 2007, 'Romania': 2007,
    'Croatia': 2013,
}
uk_membership = (1973, 2020)  # inclusive start, exclusive end

def is_eu_member(country, year):
    if country == 'United Kingdom':
        return uk_membership[0] <= year < uk_membership[1]
    return eu_accession_year.get(country, 9999) <= year

all_countries = [c for c in df_energy['country'].unique() if c not in ['EU', 'EU2', 'Non-EU']]

def aggregate_region(df, region_name, membership_fn):
    base = df[df['country'].isin(all_countries)].copy()
    is_member = base.apply(lambda r: membership_fn(r['country'], r['year']), axis=1)
    subset = base[is_member]

    grouped = subset.groupby(['year', 'source'], as_index=False).agg(
        generation_twh=('generation_twh', 'sum'),
        emissions_mtco2e=('emissions_mtco2e', 'sum')
    )

    totals = grouped[grouped['source'] == 'Total generation'][
        ['year', 'generation_twh', 'emissions_mtco2e']
    ].rename(columns={'generation_twh': 'total_gen', 'emissions_mtco2e': 'total_emissions'})

    grouped = grouped.merge(totals, on='year', how='left')
    grouped['share_pct'] = grouped['generation_twh'] / grouped['total_gen'] * 100
    grouped['share_emissions_pct'] = grouped['emissions_mtco2e'] / grouped['total_emissions'] * 100
    grouped['emissions_intensity_gco2e_kwh'] = grouped['emissions_mtco2e'] / grouped['generation_twh'] * 1000

    grouped['country'] = region_name
    grouped['iso3'] = None
    grouped['is_aggregated'] = True
    grouped = grouped.drop(columns=['total_gen', 'total_emissions'])
    return grouped[df.columns]

eu2 = aggregate_region(df_energy, 'EU2', is_eu_member)
non_eu = aggregate_region(df_energy, 'Non-EU', lambda c, y: not is_eu_member(c, y))

df_energy = df_energy[~df_energy['country'].isin(['EU2', 'Non-EU'])]
df_energy = pd.concat([df_energy, eu2, non_eu], ignore_index=True)

comparison = (
    df_energy[df_energy['country'].isin(['EU', 'EU2'])]
    .pivot_table(index=['year', 'source'], columns='country', values=numeric_cols)
)
for col in numeric_cols:
    sub = comparison[col].copy()
    sub['abs_diff'] = (sub['EU'] - sub['EU2']).abs()
    sub['pct_diff'] = (sub['abs_diff'] / sub['EU'].abs()) * 100
    print(f"\n--- {col} ---")
    print(sub.sort_values('pct_diff', ascending=False).head(5))



--- generation_twh ---
country             EU    EU2  abs_diff     pct_diff
year source                                         
2006 Net imports  0.46  14.99     14.53  3158.695652
2016 Net imports  0.65  18.40     17.75  2730.769231
2013 Net imports -1.81  12.62     14.43   797.237569
2019 Net imports  2.93  24.10     21.17   722.525597
2014 Net imports -5.03  15.49     20.52   407.952286

--- share_pct ---
country              EU       EU2  abs_diff     pct_diff
year source                                             
2006 Net imports  0.016  0.467172  0.451172  2819.823478
2016 Net imports  0.022  0.569557  0.547557  2488.896224
2013 Net imports -0.063  0.389269  0.452269   717.888075
2019 Net imports  0.102  0.752727  0.650727   637.968104
2014 Net imports -0.178  0.489881  0.667881   375.214254

--- emissions_mtco2e ---
country           EU      EU2  abs_diff   pct_diff
year source                                       
2016 Gas     210.498  279.134    68.636  32.606486
2010 Gas

In [45]:
# Pivot to get Clean and Fossil generation side by side per country/year
pivot = (
    df_energy[df_energy['source'].isin(['Clean', 'Fossil'])]
    .pivot_table(index=['country', 'year'], columns='source', values='generation_twh')
    .reset_index()
)

pivot['clean_gt_fossil'] = pivot['Clean'] > pivot['Fossil']

# For each country, find the first year where Clean surpassed Fossil
milestones = (
    pivot[pivot['clean_gt_fossil']]
    .sort_values(['country', 'year'])
    .groupby('country', as_index=False)
    .first()[['country', 'year']]
    .rename(columns={'year': 'milestone_year'})
)
milestones.to_csv("../export/country_milestones.csv", index=False)
print(milestones)

source          country  milestone_year
0               Austria            2005
1               Belgium            2005
2              Bulgaria            2005
3               Croatia            2005
4               Czechia            2020
5               Denmark            2014
6                    EU            2010
7                   EU2            2012
8               Estonia            2024
9               Finland            2005
10               France            2005
11              Germany            2019
12              Hungary            2009
13                Italy            2024
14               Latvia            2005
15            Lithuania            2005
16           Luxembourg            2016
17           Montenegro            2005
18          Netherlands            2023
19               Non-EU            2005
20      North Macedonia            2025
21               Norway            2005
22             Portugal            2010
23              Romania            2010


In [48]:
df_monthly = pd.read_csv("./release_generation_monthly_lower.csv")

keep_cols = [
    "Area", "ISO 3 code", "Date", "Electricity source", "Generation (TWh)", "Continent"
]

df_monthly = df_monthly[keep_cols].rename(columns={
    "Area": "country", "ISO 3 code": "iso3", "Date": "date",
    "Electricity source": "source",
    "Generation (TWh)": "generation_twh",
})

df_monthly['date'] = pd.to_datetime(df_monthly['date'])
df_monthly['year'] = df_monthly['date'].dt.year
df_monthly['month'] = df_monthly['date'].dt.month

keep_sources = [
    "Total generation", "Net imports", "Demand",
    "Clean", "Fossil", "Renewables", "Nuclear",
    "Wind", "Solar", "Hydro", "Bioenergy", "Other renewables",
    "Coal", "Gas", "Other fossil",
]

df_monthly = df_monthly[df_monthly["source"].isin(keep_sources)].copy()

df_monthly = df_monthly[(df_monthly['year'] >= 2015) & (df_monthly['year'] <= 2025)].reset_index(drop=True)

heatmap_data = (
    df_monthly
    .groupby(['source', 'year', 'month'], as_index=False)
    .agg(total_generated=('generation_twh', 'sum'))
)

# Get the "Total generation" value per year/month as the denominator
totals = (
    heatmap_data[heatmap_data['source'] == 'Total generation']
    [['year', 'month', 'total_generated']]
    .rename(columns={'total_generated': 'month_total'})
)

heatmap_data = heatmap_data.merge(totals, on=['year', 'month'], how='left')
heatmap_data['share_pct'] = heatmap_data['total_generated'] / heatmap_data['month_total'] * 100

heatmap_data = heatmap_data.drop(columns=['month_total'])

heatmap_data['share_pct'] = heatmap_data['share_pct'].round(3)
heatmap_data['total_generated'] = heatmap_data['total_generated'].round(3)

heatmap_data = heatmap_data.sort_values(['year', 'month', 'source']).reset_index(drop=True)
heatmap_data.to_csv("../export/source_month_generated.csv", index=False)

In [47]:
heatmap_data[heatmap_data['source'] == 'Total generation'].describe()

,year,month,total_generated,share_pct
count,132.000000,132.000000,132.000000,132.0
mean,2020.000000,6.500000,493.536129,100.0
std,3.174324,3.465203,87.861400,0.0
min,2015.000000,1.000000,223.235000,100.0
25%,2017.000000,3.750000,475.775500,100.0
50%,2020.000000,6.500000,504.345000,100.0
75%,2023.000000,9.250000,543.375750,100.0
max,2025.000000,12.000000,633.801000,100.0


In [38]:
first_year_per_country = df_monthly.groupby("country")["year"].min().sort_values(ascending=False)

print(first_year_per_country)
print()
print("Highest first year (i.e. shortest history):", first_year_per_country.index[0], "-", first_year_per_country.iloc[0])

country
Kosovo                2022
Bosnia Herzegovina    2019
Malta                 2019
Luxembourg            2018
Türkiye               2018
Croatia               2018
Serbia                2017
Switzerland           2017
North Macedonia       2017
EU                    2016
Italy                 2016
Slovenia              2016
Austria               2015
Belgium               2015
Cyprus                2015
Bulgaria              2015
France                2015
Germany               2015
Greece                2015
Czechia               2015
Denmark               2015
Estonia               2015
Latvia                2015
Lithuania             2015
Ireland               2015
Hungary               2015
Finland               2015
Norway                2015
Netherlands           2015
Montenegro            2015
Romania               2015
Portugal              2015
Poland                2015
Slovakia              2015
Sweden                2015
Spain                 2015
Ukraine             

In [39]:
last_year_per_country = df_monthly.groupby("country")["year"].max().sort_values(ascending=True)

print(last_year_per_country)
print()
print("Lowest last year (first to end):", last_year_per_country.index[0], "-", last_year_per_country.iloc[0])

country
Ukraine               2022
Austria               2025
Bosnia Herzegovina    2025
Belgium               2025
Croatia               2025
Cyprus                2025
Czechia               2025
Bulgaria              2025
EU                    2025
Estonia               2025
Finland               2025
France                2025
Germany               2025
Greece                2025
Hungary               2025
Denmark               2025
Italy                 2025
Kosovo                2025
Latvia                2025
Lithuania             2025
Luxembourg            2025
Malta                 2025
Montenegro            2025
Netherlands           2025
North Macedonia       2025
Norway                2025
Poland                2025
Portugal              2025
Romania               2025
Serbia                2025
Slovakia              2025
Ireland               2025
Slovenia              2025
Spain                 2025
Switzerland           2025
Sweden                2025
Türkiye             